### **Student Information**
GitHub ID:

Kaggle name:

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

# **Project Report**

## 1. Model Development (10 pts Required)

### 1.0 Overview

I tried three routes sequentially and submitted four models:

| Route | Method | Mean macro-F1 (5-fold CV) | Private leaderboard score |
| --- | --- | --- | --- |
| 1 | TF-IDF + LinearSVC | 0.5219 | 0.5220 |
| 2 (initial) | BERTweet fine-tuning: LR=3e-5, unweighted CE | 0.5947 | 0.6055 |
| 2 (updated) | BERTweet fine-tuning: LR=4e-5, label smoothing=0.05 | 0.6048 | 0.6187 |
| 3 | Hartmann RoBERTa-large fine-tuning | 0.6205 | 0.6456 |

The Route 3 model, fine-tuned from Hartmann RoBERTa-large, achieved the best Kaggle score among my four submissions.

### 1.1 Preprocessing Steps

- I kept only the first entry for each identical text-label combination. If the same text had different labels, I kept one entry for each label rather than choosing one label myself. This left 3,326 training samples.

- I used `StratifiedGroupKFold` for five-fold cross-validation. In each fold, roughly 80% of the data was used for training and 20% for validation. I grouped samples by their exact original text so that the same sentence could not appear in both sets. All three routes used the same saved folds.

- I normalized Unicode using NFC and cleaned up extra whitespace.

- For the SVM route, I removed mentions, URLs, and `[NAME]`. I used TweetTokenizer, with extra handling for emoticons, to keep emoji and other emotional symbols, then lowercased the word tokens. I also tested stopword removal and lemmatization.

- For the BERTweet route, I replaced mentions, URLs, and `[NAME]` with `@USER`, `HTTPURL`, and `person`. I kept capitalization, stopwords, and original word forms. The BERTweet tokenizer then normalized the text, converted emoji into text descriptions, and split the text into subword tokens.

- For the Hartmann route, I kept the original mentions, URLs, `[NAME]`, capitalization, and emoji, and used its own tokenizer. BERTweet-specific normalization was not applied. Both Transformer routes used a maximum length of 128 tokens and dynamic padding.

### 1.2 Feature Engineering Steps

- The SVM model combined word unigram/bigram and character 3–5 gram TF-IDF features. I applied L2 normalization to each branch separately, concatenated them, and passed the result to LinearSVC. Stopword removal and lemmatization applied only to the word branch. The vocabulary and IDF were fitted only on the training portion of each fold.

- The BERTweet and Hartmann models used their own pretrained tokenizers to convert text into subword token IDs. Their Transformer encoders then learned contextual representations. The submitted models used the final-layer representation of the first special token, `<s>`, for classification.

### 1.3 Explanation of Your Model

| Route | Architecture | Explanation |
| --- | --- | --- |
| 1 | TF-IDF → LinearSVC classifier | After ablation experiments and hyperparameter tuning, the best configuration combined word unigram/bigram, character 3–5 gram, emoji retention, stopword removal, and lemmatization. It used `C=0.1`, `class_weight="balanced"`, `min_df=1`, `max_df=0.9`, `sublinear_tf=False`, and no limit on `max_features`. |
| 2 | BERTweet embedding + 12 Transformer encoder layers → final `<s>` vector → Dropout → Linear(768, 768) → tanh → Dropout → Linear(768, 6) | I compared learning rates and class weights. I also tested mean and attention pooling, but neither outperformed CLS pooling in this comparison. The initial submission used LR=3e-5 and unweighted cross-entropy. The updated version used LR=4e-5, unweighted cross-entropy, label smoothing=0.05, and seed=42. |
| 3 | Hartmann RoBERTa-large encoder → final `<s>` vector → existing classification MLP → new six-class output layer | I kept the pretrained encoder and intermediate classification layers, replaced the original seven-class output layer with a newly initialized six-class layer, and fine-tuned all parameters. I used LR=4e-5, unweighted cross-entropy, label smoothing=0.05, and seed=42. Among the completed Route 3 candidates, this model had the highest mean validation macro-F1. |

- Error analysis showed that disgust was often misclassified as anger. I then tested higher learning rates and label smoothing to see whether they improved validation performance.

- Both Transformer routes fine-tuned the embeddings, encoder, and classification head for five epochs.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

The tables below list the prompts I kept from my route1, route2, and route3 chats, in chronological order within each route. Quoted context is included when a prompt referred to an earlier answer.


#### Route 1

| AI prompt | My decision and reason |
| --- | --- |
| **R1-01**<br><br>**Prompt:**<br>check homework.ipynb。my goal is to effeciently achieve high F1 on prediction。Do u think that i should directly finetune Bert language model，or try TFiDF+SVM/GBDT, word2vec+LSTM, these two basic ML structures first。give a clear explanation on the three road in mandarin | I expected BERT-base to perform best, but wanted to understand how the other approaches would be implemented and whether they were worth trying. After considering the AI's analysis and suggestions, I decided to build a low-cost SVM baseline before fine-tuning a pretrained model. I did not use Word2Vec + LSTM because the corpus was too small to learn reliable word meanings, and the many parameters to tune might not justify the effort. |
| **R1-02**<br><br>**Prompt:**<br>I agree that i'll try **TF-IDF＋LinearSVC first. I don't know if you remember, I trained a model that predict the popularity of the news recently, and it turns out that gbdt outperformed LinearSVM, or you just expext that Bert would be better any way so doing the baseline does not matter. answer in mandarin** | I understood that the SVM baseline would provide a benchmark, help check the data pipeline, and let me track improvements in later models. Since the data lacked metadata, I thought GBDT was unlikely to gain much advantage here. |
| **R1-03**<br><br>**Prompt:**<br>Moreover, i just took a glance at the data, it seems that the text data are quite neat and there aren't any metadata, so it does not require beatifulsoup for html right? Also, I noticed that there are some useful symbol such as :) ,  B-) and emojiI like 😂, but there are also a lot of useless ones such as @\&lt;username&gt; and  [NAME]. Check the data and tell me how would you preprocess the data for this baseline. cleaned  unigram/bigram TFIDF? I think we should also do these  ablation: stopword and lemmatization, hp tuning (min_df, max_features?) or use fpgrowth on tfidf? | I clarified how to handle symbols such as emoji and confirmed the experiment order. The AI's suggestions matched my ablation plan, including stopword removal, lemmatization, and hyperparameter tuning. I also realized that FP-Growth finds frequent patterns without using class labels, so I decided not to use it for this classification task. I had initially confused its purpose. |
| **R1-04**<br><br>**Quoted context:**<br>有兩種可行做法：<br><br>使用能識別社群文本、emoji 和 emoticon 的 tokenizer，再產生 word n-gram。<br>先將表情轉成不會被預設 tokenizer 丟棄的 token，例如 :) → EMO_SMILE、`:( → EMO_SAD`、`😂 → EMO_TEARS_OF_JOY`。<br>我較傾向保留不同表情的區別，不要把所有 emoji 都變成同一個 EMOJI。不過，表情數量不多，也不要先花大量時間手寫複雜規則；可以先跑基本模型，再驗證增加表情特徵是否有用。<br><br>**Prompt:**<br>briefly explain what these two method differs in the following process and your preference | I followed the AI's suggestion to use a tokenizer designed for social media text because it could preserve the original emoticons without requiring me to assign an emotional meaning to each symbol in advance. |
| **R1-05**<br><br>**Quoted context:**<br>加入 character n-gram 特徵<br><br>**Prompt:**<br>different with unigram/bigram? in brief | I learned that character n-grams capture sequences of consecutive characters and can help represent spelling variations in tweets. |
| **R1-06**<br><br>**Prompt:**<br>Ok , remove duplicated text to 1 left, split dev data to stratified 5 fold cross validation. BTW, it's mean F1. Implement the route1.ipynb (automatically record the result in run.csv) and other useful py file to conduct these experiments based on our conclusion in the conversation. Do everything in lab_competition. Write comments in the code. Also present a md in plain mandarin language that report your implementation. | - |
| **R1-07**<br><br>**Prompt:**<br>keep stage,  name, those f1s and config_json only in run.csv | - |
| **R1-08**<br><br>**Prompt:**<br>oops, the csv i wanted to simplify is leaderboard.csv. simplify the existing leaderboard.csv as well | - |
| **R1-09**<br><br>**Prompt:**<br>i want all original information in run.csv, the one I want to simplify is leaderboard.csv. Also, edit the existing leaderboard.csv to the simplified version i desire. | - |
| **R1-10**<br><br>**Prompt:**<br>The result seems bad, none of a model reached a F1 &gt; 0.5. Investigate that is there anything that is done wrongly? or what limit the performance of these models predicting emotion tags? should i move to route2 that finetune BERT? | I understood that the main problems were limited generalization and ambiguity in the data. I agreed with the AI's suggestion to run a few more rounds of hyperparameter tuning, paying attention to the weak performance on disgust. After assessing this route, I felt that further work on it would offer limited benefit. |
| **R1-11**<br><br>**Quoted context:**<br>針對 word＋char 調整 C 和 class weight，再比較加入 stopword 處理的結果。不建議繼續投入大量清理規則<br><br>**Prompt:**<br>ok do it. you should try combining thing that was noticeably helpful based on the ablation experiments and then tune the hp. no need to produce new md in this section only update | - |
| **R1-12**<br><br>**Prompt:**<br>We're only keeping the 1st text on duplicated samples. I want to revise the rule that: keep both if their tags are different, only keep the 1st one if their tags are same. Then, use text as the grouping unit for stratified group five-fold cross-validation: the same text group must be entirely in training, or entirely in validation. Repeat the experiment on the current best model and tune hp using this new strategy. | I changed the deduplication rule to what I considered more reasonable: keep only the first sample for each identical text-label pair, but retain samples with the same text and different labels. I tested the previous best configuration under this rule, and the result was similar. |
| **R1-13**<br><br>**Prompt:**<br>you did not update the report.. (plz update it)And, do you think my data preprocessing strategy is now appropriate and complete? | The AI pointed out two remaining limitations: identical text could have conflicting labels, and grouping was based only on exact original-text matches. I considered these limitations but kept the current approach. |


#### Route 2

| AI prompt | My decision and reason |
| --- | --- |
| **R2-01**<br><br>**Prompt:**<br>Ok, this conversation we'll try the route 3 using **BERTweet from. Check** [@比较文本分类模型路线](thread://01a1fecc-3ddd-7b13-a01c-dabf3d80b3af?hostId=local) about what are done. It automatically deal with tokenization, right? what would change about preprocessing from the previous LinearSVC model? Is the architecture provided here [https://chatgpt.com/c/6b02086d-0c28-83ec-b78a-463d78eccdb0](https://chatgpt.com/c/6b02086d-0c28-83ec-b78a-463d78eccdb0) a suitable baseline (RoBERTa Classification Head)? Should i add class weights on loss computation to deal with calss imbalance? ans in mandarin for all query | I learned how BERTweet's tokenizer works and how its preprocessing differs from the LinearSVC baseline. After briefly checking the official documentation, I decided to use it and gained a better understanding of the model architecture. I also agreed that class weights were worth testing in a later ablation to see whether they helped with class imbalance. |
| **R2-02**<br><br>**Quoted context:**<br>建議第一版一起微調 BERTweet encoder 與分類頭。分類頭是新初始化的，需要用你的六類標籤訓練。<br><br>BERTweet baseline<br><br>**Prompt:**<br>So we're fintuning all the parameters in the embedding, 12 transformer encoder and classification head, right? and briefly explain why BERTweet baseline keep or discard | I learned the reasons behind BERTweet's preprocessing choices. For example, emoji information can be retained through the text normalization used during pretraining, rather than being removed. |
| **R2-03**<br><br>**Quoted context:**<br>BERTweet 流程文字化<br><br>**Prompt:**<br>briefly explain how | I learned that BERTweet converts emoji into English text descriptions. |
| **R2-04**<br><br>**Prompt:**<br>ok implement BERTweet baseline (follow the config attached) in route2.ipynb based on the conclusion we discussed, make sure it support weighted loss and lr tuning ablation. write implementation_route2.md in mandarin. share the two csv with route1. Don't train anything yet. I'll get a colab gpu for training. | - |
| **R2-05**<br><br>**Prompt:**<br>`RobertaClassificationHead` is CLS Pooling right? should i try mean pooling and attention pooling? why? if yes how would you propose the experiment? | I learned that mean pooling averages the valid token vectors and may dilute the contribution of a few strongly emotional words, while attention pooling learns a weight for each token and takes a weighted average. I thought both were worth trying, so I decided to compare pooling methods while keeping the same classification head, using a simple linear layer followed by softmax for attention pooling. |
| **R2-06**<br><br>**Quoted context:**<br>排除 &lt;s&gt; 和 &lt;/s&gt;，CLS 則取 &lt;s&gt;<br><br>**Prompt:**<br>briefly explain why excluding \&lt;s&gt; and \&lt;/s&gt; | The AI explained that excluding the special boundary tokens from mean pooling keeps the average focused on the tokens in the post itself. |
| **R2-07**<br><br>**Prompt:**<br>OK, I have done CLS PLAN tuning. implement the proposed next experiments in notebook. attention pooling layer use linear-&gt;softmax. Also, print the config or what is being trained when training so I can quickly review the progress. Do not train. | - |
| **R2-08**<br><br>**Prompt:**<br>i think the result shows CLS pooling is still better. how can i further improve? | The AI suggested error analysis, which I agreed with because my past experience showed that focusing on a model's weaknesses often helped. I also decided to continue tuning the parameters and try multiple seeds to check stability and whether an ensemble would help. |
| **R2-09**<br><br>**Quoted context:**<br>優先做一輪小型 CLS 實驗<br><br>**Prompt:**<br>ok do the error analysis and report, then implement the CLS experiment you proposed but don't train | - |
| **R2-10**<br><br>**Prompt:**<br>looks like bertweet_cls_lr=4e-05_weighted=False_smooth=0.05_seed=42 is currently the best. should i try ensembles? | I did not use a multi-seed ensemble because the improvement was very small. |


#### Route 3

| AI prompt | My decision and reason |
| --- | --- |
| **R3-01**<br><br>**Prompt:**<br>I'd would like to try these existing models. What's the difference between the current model (CLS pooling)? How would you design the experiment? ans in mandarin. | Hartmann DistilRoBERTa and Hartmann RoBERTa-large are smaller and larger RoBERTa models that have already been fine-tuned on emotion data. Like BERTweet, they use the first special token for classification. Although their original task has seven emotion classes, they can be used for direct prediction. Cardiff Twitter RoBERTa-large is another large emotion model, but its original task has 11 labels. I decided to first try direct inference by choosing the highest-scoring emotion among the six target classes. I then retained the pretrained encoder and intermediate classification layers, replaced the final output layer with a newly initialized six-class layer, and trained all parameters with single-label cross-entropy. This let me use the pretrained knowledge while adapting the model to this task. |
| **R3-02**<br><br>**Quoted context:**<br>先確認外部資料是否重疊<br><br>**Prompt:**<br>check this and explain how you did | To investigate possible data leakage, the AI checked the model documentation and compared normalized competition text against public datasets for exact matches. I considered the overlap findings a potential risk, but still decided to try these models. |
| **R3-03**<br><br>**Prompt:**<br>Ok then still implement route3.ipynb for these tasks | - |
| **R3-04**<br><br>**Prompt:**<br>The baseline config in this set of experiment  from route2.ipynb's best setting, right? | I treated Route 3 as an extension of Route 2 and used Route 2's best configuration as the default. |
| **R3-05**<br><br>**Prompt:**<br>the training appeared an error after finishing C, check the ipynb. Can i still generate submission? | - |

---

# **Competition Code**

The following cells show the core implementation used for the four submissions, organized into preprocessing, feature engineering, and model implementation. The code is condensed from the original files: experiment logging, caching, full search loops, and unused model branches are omitted.


## 1. Preprocessing Steps

### Data preparation and shared folds

Source: `lab_competition/route1.py` — `prepare_data()`; `lab_competition/route2.py` — `load_data()`. Condensed to retain data merging, text-label deduplication, and grouped five-fold validation. Saved folds are reused when available.

In [ ]:
# Sources: route1.py / prepare_data() and route2.py / load_data().
from pathlib import Path
from dataclasses import dataclass
import gc
import json
import re
import unicodedata

import nltk
import numpy as np
import pandas as pd
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

WORK = Path("lab_competition")  # Change this path when using Colab / Drive.
nltk.data.path.insert(0, str(WORK / "nltk_data"))
SEED = 42
LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}

def prepare_data():
    posts = pd.read_json(WORK / "data/final_posts.json")
    labels = pd.read_csv(WORK / "data/emotion.csv", keep_default_na=False)
    sample = pd.read_csv(WORK / "data/sampleSubmission.csv", keep_default_na=False)
    assert posts.post_id.is_unique and labels.post_id.is_unique and sample.post_id.is_unique
    train = labels.merge(posts, on="post_id", validate="one_to_one", sort=False)
    # Keep conflicting labels, but remove repeated copies of the same text-label pair.
    train = train.drop_duplicates(["text", "label"]).reset_index(drop=True)
    test = sample[["post_id"]].merge(
        posts, on="post_id", how="left", validate="one_to_one", sort=False,
    )
    assert train.text.notna().all() and test.text.notna().all()
    assert set(train.label) == set(LABELS)
    assert test.post_id.tolist() == sample.post_id.tolist()

    saved_path = WORK / "artifacts/train_folds.csv"
    if saved_path.is_file():
        saved = pd.read_csv(saved_path, keep_default_na=False)
        pd.testing.assert_frame_equal(
            train[["post_id", "text", "label"]], saved[["post_id", "text", "label"]],
            check_dtype=False,
        )
        train["fold"] = saved.fold.to_numpy()
    else:
        # Group by exact raw text before any model-specific cleaning.
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
        train["fold"] = -1
        for fold, (_, dev) in enumerate(splitter.split(train.text, train.label, groups=train.text)):
            train.loc[dev, "fold"] = fold
    assert set(train.fold) == set(range(5))
    assert train.groupby("text").fold.nunique().max() == 1
    return train, test


### Route 1: cleaning and word tokenization

Source: `lab_competition/route1.py` — `clean_text()` and `WordTokenizer`. These functions retain the original emoticon protection, negation-preserving stopword filter, and POS-based WordNet lemmatization. Lemmatization requires the original NLTK WordNet and English POS-tagger resources.

In [ ]:
# Source: lab_competition/route1.py (same cleaning and tokenization rules).
EMOTICON = re.compile(r"(?:[B8]-[)(/\\DPp]|[:;=][\-^']?[)(/\\DPp]|<3|(?<!\w)xd(?!\w))", re.I)
NEGATIONS = {"no", "not", "nor", "never", "cannot", "neither", "none", "nobody", "nothing", "nowhere"}
STOPWORDS = set(ENGLISH_STOP_WORDS) - NEGATIONS


def clean_text(text, placeholders="remove"):
    """Normalize lightly without discarding sentiment-bearing symbols."""
    text = unicodedata.normalize("NFC", str(text)).replace("’", "'").replace("‘", "'")
    user = " USER " if placeholders == "replace" else " "
    person = " PERSON " if placeholders == "replace" else " "
    url = " URL " if placeholders == "replace" else " "
    text = re.sub(r"@<username>|@\w+", lambda m: user, text, flags=re.I)
    text = re.sub(r"\[NAME\]", lambda m: person, text, flags=re.I)
    text = re.sub(r"#+URL#+|https?://\S+|www\.\S+", lambda m: url, text, flags=re.I)
    # Preserve hashtag words; do not strip arbitrary bracketed content.
    text = re.sub(r"#(?=\w)", "", text)
    return re.sub(r"\s+", " ", text).strip()



class WordTokenizer:
    def __init__(self, mode="tweet", symbols=True, stopwords=False, lemmatize=False):
        self.mode = mode
        self.symbols = symbols
        self.stopwords = stopwords
        self.lemmatize = lemmatize

    def __call__(self, text):
        if self.mode == "regex":
            tokens = re.findall(r"\b\w\w+\b", text.lower())
        else:
            tokenizer = TweetTokenizer(preserve_case=True, reduce_len=False, strip_handles=False)
            # Protect emoticons such as B-), which TweetTokenizer otherwise splits.
            tokens = []
            end = 0
            for match in EMOTICON.finditer(text):
                tokens.extend(tokenizer.tokenize(text[end:match.start()]))
                tokens.append(match.group())
                end = match.end()
            tokens.extend(tokenizer.tokenize(text[end:]))
            tokens = [token.lower() for token in tokens]
        if not self.symbols:
            tokens = [token for token in tokens if any(c.isalnum() for c in token) and not EMOTICON.fullmatch(token)]
        if self.stopwords:
            # Contractions containing n't are never removed by this filter.
            tokens = [token for token in tokens if token not in STOPWORDS]
        if self.lemmatize and tokens:
            from nltk.stem import WordNetLemmatizer

            lemma = WordNetLemmatizer()
            tags = nltk.pos_tag(tokens, lang="eng")
            pos_map = {"J": "a", "V": "v", "N": "n", "R": "r"}
            tokens = [lemma.lemmatize(token, pos_map[tag[0]]) if token.isalpha() and tag[0] in pos_map else token for token, tag in tags]
        return tokens


### Routes 2 and 3: Transformer input text

Sources: `lab_competition/route2.py` — `prepare_text()`; `lab_competition/route3.py` — `prepare_text()`. Only the submitted BERTweet-base and Hartmann branches are retained.

In [ ]:
# Sources: route2.py / prepare_text() and route3.py / prepare_text().
def prepare_transformer_text(text, route):
    text = unicodedata.normalize("NFC", str(text))
    if route == "bertweet":
        text = re.sub(r"@<username>", "@USER", text, flags=re.I)
        text = re.sub(r"#+URL#+", "HTTPURL", text, flags=re.I)
        text = re.sub(r"\[NAME\]", "person", text, flags=re.I)
    # Hartmann keeps mentions, placeholders, case, punctuation, and emoji.
    return re.sub(r"\s+", " ", text).strip()


## 2. Feature Engineering Steps

### Word and character TF-IDF

Source: `lab_competition/route1.py` — `Config` and `build_pipeline()`. The final configuration is from `submission_info.json`. The word branch applies stopword removal and lemmatization; the character branch uses lightly cleaned text. Each branch uses L2 normalization.

In [ ]:
# Source: lab_competition/route1.py; final settings: submission_info.json.
@dataclass(frozen=True)
class SVMConfig:
    tokenizer: str = "tweet"
    word_max_n: int = 2
    symbols: bool = True
    placeholders: str = "remove"
    stopwords: bool = False
    lemmatize: bool = False
    char_features: bool = False
    min_df: int = 1
    max_df: float = 1.0
    max_features: int | None = None
    sublinear_tf: bool = True
    C: float = 1.0
    class_weight: str | None = None



def build_pipeline(config):
    from functools import partial

    word = TfidfVectorizer(
        preprocessor=partial(clean_text, placeholders=config.placeholders), lowercase=False,
        tokenizer=WordTokenizer(config.tokenizer, config.symbols, config.stopwords, config.lemmatize),
        token_pattern=None, ngram_range=(1, config.word_max_n), min_df=config.min_df,
        max_df=config.max_df, max_features=config.max_features,
        sublinear_tf=config.sublinear_tf, norm="l2", dtype=np.float64,
    )
    features = word
    if config.char_features:
        # char preserves punctuation and cross-word fragments; char_wb would differ.
        char = TfidfVectorizer(
            preprocessor=partial(clean_text, placeholders=config.placeholders),
            analyzer="char", lowercase=False, ngram_range=(3, 5),
            min_df=config.min_df, max_df=config.max_df, max_features=config.max_features,
            sublinear_tf=config.sublinear_tf, dtype=np.float64,
        )
        features = FeatureUnion([("word", word), ("char", char)])
    return Pipeline([
        ("tfidf", features),
        ("svm", LinearSVC(C=config.C, class_weight=config.class_weight, random_state=SEED, max_iter=10000)),
    ])



SVM_CONFIG = SVMConfig(
    char_features=True, stopwords=True, lemmatize=True,
    C=0.1, class_weight="balanced", min_df=1, max_df=0.9,
    max_features=None, sublinear_tf=False,
)


### Transformer tokenization and pooling

Sources: `lab_competition/route2.py` — tokenization in `evaluate()`; `lab_competition/route2_models.py` — `pool()`. The submissions use the native CLS head. The compact pooling function also shows the mean and linear-attention alternatives tested in Route 2; it does not replace the submitted CLS models.

In [ ]:
# Source: lab_competition/route2.py / evaluate(), shared with the Hartmann route.
def tokenize_posts(texts, config):
    from transformers import AutoTokenizer
    options = {"normalization": True} if config["route"] == "bertweet" else {}
    tokenizer = AutoTokenizer.from_pretrained(config["model"], use_fast=False, **options)
    texts = [prepare_transformer_text(text, config["route"]) for text in texts]
    encoded = tokenizer(texts, truncation=True, max_length=config["max_length"], padding=False)
    # Padding is applied dynamically to each batch by DataCollatorWithPadding.
    return tokenizer, encoded


# Source: lab_competition/route2_models.py / pool() (condensed).
def pool_tokens(hidden, input_ids, attention_mask, model_config, method="cls", score_layer=None):
    if method == "cls":
        return hidden[:, 0]
    mask = attention_mask.bool()
    mask = mask & input_ids.ne(model_config.bos_token_id) & input_ids.ne(model_config.eos_token_id)
    if not mask.any(dim=1).all():
        raise ValueError("Pooling requires at least one content token.")
    if method == "mean":
        return (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)
    # Attention uses a trainable nn.Linear(hidden_size, 1), followed by masked softmax.
    if method != "attention" or score_layer is None:
        raise ValueError("Attention pooling requires its linear scoring layer.")
    scores = score_layer(hidden).squeeze(-1).float().masked_fill(~mask, -float("inf"))
    weights = scores.softmax(dim=1).to(hidden.dtype)
    return (hidden * weights.unsqueeze(-1)).sum(1)


## 3. Model Implementation Steps

### Route 1: five-fold evaluation and final refit

Source: `lab_competition/route1.py` — `evaluate()` and `export_submission()`. Logging and parameter search are removed. TF-IDF is fitted only within each training fold; the final SVM is refitted on all deduplicated training samples.

In [ ]:
# Source: lab_competition/route1.py / evaluate() and export_submission() (condensed).
def train_svm_cv(train):
    scores = []
    for fold in range(5):
        fit, dev = train.fold != fold, train.fold == fold
        model = build_pipeline(SVM_CONFIG)
        model.fit(train.loc[fit, "text"], train.loc[fit, "label"])
        predictions = model.predict(train.loc[dev, "text"])
        scores.append(f1_score(train.loc[dev, "label"], predictions, average="macro", zero_division=0))
    print("SVM mean macro-F1:", np.mean(scores))
    final_model = build_pipeline(SVM_CONFIG)
    final_model.fit(train.text, train.label)
    return final_model


### Routes 2 and 3: classification heads and full fine-tuning

Sources: `lab_competition/route2.py` — `evaluate()`; `lab_competition/route3.py` — `load_finetune_model()` and `evaluate()`. The common loop retains full fine-tuning, optional fold-specific class weights, label smoothing, dynamic padding, and best-validation-epoch selection. The Hartmann branch preserves its pretrained intermediate head and replaces the seven-class output with six classes.

In [ ]:
# Sources: route2.py / evaluate() and route3.py / load_finetune_model().
def load_classifier(config):
    import torch
    from transformers import AutoConfig, AutoModelForSequenceClassification
    model_config = AutoConfig.from_pretrained(config["model"])
    model_config.hidden_dropout_prob = 0.1
    model_config.attention_probs_dropout_prob = 0.1
    model_config.classifier_dropout = 0.1
    if config["route"] == "bertweet":
        model_config.num_labels = len(LABELS)
    model = AutoModelForSequenceClassification.from_pretrained(config["model"], config=model_config)
    if config["route"] == "hartmann":
        # Retain the encoder and intermediate MLP; initialize only the six-class output.
        model.classifier.out_proj = torch.nn.Linear(model.classifier.out_proj.in_features, len(LABELS))
        model._init_weights(model.classifier.out_proj)
        model.classifier.num_labels = len(LABELS)
    model.num_labels = model.config.num_labels = len(LABELS)
    model.config.label2id = LABEL2ID
    model.config.id2label = dict(enumerate(LABELS))
    model.config.problem_type = "single_label_classification"
    assert all(parameter.requires_grad for parameter in model.parameters())
    return model


# Sources: route2.py / evaluate() and route3.py / evaluate() (condensed).
def train_transformer_cv(train, name, config):
    import torch
    from transformers import DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
    tokenizer, encoded = tokenize_posts(train.text.tolist(), config)
    labels = np.array([LABEL2ID[label] for label in train.label])

    class TextDataset(torch.utils.data.Dataset):
        def __init__(self, indices):
            self.indices = indices
        def __len__(self):
            return len(self.indices)
        def __getitem__(self, index):
            i = int(self.indices[index])
            return {**{key: values[i] for key, values in encoded.items()}, "labels": int(labels[i])}

    class LossTrainer(Trainer):
        def __init__(self, *args, weights=None, **kwargs):
            super().__init__(*args, **kwargs)
            self.class_weights = weights
            # Trainer handles gradient-accumulation scaling, as in the original Hartmann loop.
            self.model_accepts_loss_kwargs = False
        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            inputs = dict(inputs)
            targets = inputs.pop("labels")
            outputs = model(**inputs)
            weights = None if self.class_weights is None else self.class_weights.to(outputs.logits.device)
            loss = torch.nn.functional.cross_entropy(
                outputs.logits.float(), targets, weight=weights,
                label_smoothing=config["label_smoothing"],
            )
            return (loss, outputs) if return_outputs else loss

    def metrics(prediction):
        logits, targets = prediction
        return {"macro_f1": f1_score(targets, logits.argmax(-1), average="macro", zero_division=0)}

    scores, checkpoints = [], []
    for fold in range(5):
        set_seed(SEED)
        fit = np.flatnonzero(train.fold.to_numpy() != fold)
        dev = np.flatnonzero(train.fold.to_numpy() == fold)
        weights = None
        if config["weighted_loss"]:
            counts = np.bincount(labels[fit], minlength=len(LABELS))
            weights = torch.tensor(len(fit) / (len(LABELS) * counts), dtype=torch.float32)
        model = load_classifier(config)
        # A separate folder avoids overwriting the original experiment artifacts.
        folder = WORK / "core_checkpoints" / name / f"fold{fold}"
        args = TrainingArguments(
            output_dir=str(folder), learning_rate=config["learning_rate"],
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=config["batch_size"],
            gradient_accumulation_steps=config["accumulation"], num_train_epochs=5,
            gradient_checkpointing=config["gradient_checkpointing"],
            weight_decay=0.01, warmup_ratio=0.1, lr_scheduler_type="linear", optim="adamw_torch",
            eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
            load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
            seed=SEED, data_seed=SEED, fp16=False, bf16=False,
            report_to="none", logging_steps=25, dataloader_num_workers=0,
        )
        trainer = LossTrainer(
            model=model, args=args, train_dataset=TextDataset(fit), eval_dataset=TextDataset(dev),
            processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
            compute_metrics=metrics, weights=weights,
        )
        trainer.train()
        scores.append(trainer.predict(TextDataset(dev)).metrics["test_macro_f1"])
        checkpoint = folder / "best_model"
        trainer.save_model(str(checkpoint))
        tokenizer.save_pretrained(str(checkpoint))
        checkpoints.append(checkpoint)
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()
    print(name, "mean macro-F1:", np.mean(scores))
    return checkpoints


### Four submission settings and prediction

Sources: final configurations in `lab_competition/route1.ipynb`, `route2.ipynb`, and `route3.ipynb`; five-fold probability averaging in `route2_submission.py` and `route3_submission.py`. This keeps the initial and updated BERTweet settings separate. No multi-seed ensemble is used.

In [ ]:
# Sources: final configurations in route2.ipynb and route3.ipynb.
COMMON = dict(max_length=128, weighted_loss=False)
TRANSFORMER_CONFIGS = {
    "route2_initial": dict(COMMON, route="bertweet", model="vinai/bertweet-base",
        learning_rate=3e-5, label_smoothing=0.0, batch_size=32, accumulation=1, gradient_checkpointing=False),
    "route2_updated": dict(COMMON, route="bertweet", model="vinai/bertweet-base",
        learning_rate=4e-5, label_smoothing=0.05, batch_size=32, accumulation=1, gradient_checkpointing=False),
    "route3": dict(COMMON, route="hartmann", model="j-hartmann/emotion-english-roberta-large",
        learning_rate=4e-5, label_smoothing=0.05, batch_size=4, accumulation=8, gradient_checkpointing=True),
}


# Sources: route2_submission.py and route3_submission.py / generate_submission().
def predict_five_folds(checkpoints, test, config):
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    assert len(checkpoints) == 5
    texts = [prepare_transformer_text(text, config["route"]) for text in test.text]
    probabilities = np.zeros((len(test), len(LABELS)), dtype=np.float64)
    options = {"normalization": True} if config["route"] == "bertweet" else {}
    for checkpoint in checkpoints:
        tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=False, local_files_only=True, **options)
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint, local_files_only=True).to("cuda")
        assert model.config.label2id == LABEL2ID
        model.eval()
        with torch.inference_mode():
            for start in range(0, len(texts), 32):
                inputs = tokenizer(texts[start:start + 32], padding=True, truncation=True,
                                   max_length=config["max_length"], return_tensors="pt")
                inputs = {key: value.to("cuda") for key, value in inputs.items()}
                probs = model(**inputs).logits.float().softmax(-1).cpu().numpy()
                probabilities[start:start + len(probs)] += probs / 5
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
    return np.array(LABELS)[probabilities.argmax(1)]


# Source: submission exporters; Kaggle requires the identifier column to be named id.
def save_submission(test, predictions, filename):
    assert len(predictions) == len(test) and set(predictions).issubset(LABELS)
    output = pd.DataFrame({"id": test.post_id.to_numpy(), "label": predictions})
    output.to_csv(WORK / filename, index=False)


# Explicit switches keep Run All from training or generating predictions.
RUN_SVM = False
RUN_TRANSFORMERS = False
WRITE_SUBMISSIONS = False
if RUN_SVM or RUN_TRANSFORMERS:
    train, test = prepare_data()
if RUN_SVM:
    svm = train_svm_cv(train)
    if WRITE_SUBMISSIONS:
        save_submission(test, svm.predict(test.text), "submission.csv")
if RUN_TRANSFORMERS:
    filenames = {"route2_initial": "submission_route2.csv",
                 "route2_updated": "submission_route2-2.csv", "route3": "submission_route3.csv"}
    for name, config in TRANSFORMER_CONFIGS.items():
        checkpoints = train_transformer_cv(train, name, config)
        if WRITE_SUBMISSIONS:
            save_submission(test, predict_five_folds(checkpoints, test, config), filenames[name])
